In [1]:
# CELL 1 — imports & paths

from pathlib import Path
import pandas as pd
import numpy as np
from itertools import combinations
from tqdm.auto import tqdm

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

DATA_DIR = Path("../data/full_dedup")

DEDUP_PATH = DATA_DIR / "deduplicated_records.parquet"
ENTITIES_PATH = DATA_DIR / "entities.parquet"

deduplicated = pd.read_parquet(DEDUP_PATH)
entities = pd.read_parquet(ENTITIES_PATH)

print(deduplicated.shape)
print(entities.shape)

(3653581, 18)
(1709377, 18)


In [2]:
# CELL 2 — cluster stats

cluster_stats = (
    deduplicated
    .groupby("cluster_id")
    .agg(
        cluster_size=("cluster_id", "size"),
        unique_names=("name_latin", "nunique"),
        unique_countries=("country", "nunique"),
    )
    .reset_index()
)

cluster_stats["is_suspicious"] = (
    (cluster_stats["cluster_size"] > 20)
    | (cluster_stats["unique_names"] > 5)
    | (cluster_stats["unique_countries"] > 1)
)

display(cluster_stats.describe())
display(cluster_stats["is_suspicious"].value_counts())
display(cluster_stats.sort_values("cluster_size", ascending=False).head(20))

,cluster_id,cluster_size,unique_names,unique_countries
count,1.709377e+06,1.709377e+06,1.709377e+06,1709377.0
mean,8.546880e+05,2.137376e+00,1.134936e+00,1.0
std,4.934548e+05,5.422571e+00,3.798145e-01,0.0
min,0.000000e+00,1.000000e+00,1.000000e+00,1.0
25%,4.273440e+05,1.000000e+00,1.000000e+00,1.0
50%,8.546880e+05,1.000000e+00,1.000000e+00,1.0
75%,1.282032e+06,2.000000e+00,1.000000e+00,1.0
max,1.709376e+06,4.930000e+02,4.700000e+01,1.0


is_suspicious
False    1701239
True        8138
Name: count, dtype: int64

,cluster_id,cluster_size,unique_names,unique_countries,is_suspicious
376991,376991,493,1,1,True
3110,3110,492,1,1,True
165,165,490,1,1,True
305,305,485,1,1,True
174,174,478,1,1,True
989,989,473,1,1,True
3286,3286,470,1,1,True
33,33,468,1,1,True
871,871,462,1,1,True
257,257,459,1,1,True


In [3]:
# CELL 3 — positive pairs from same cluster

N_POSITIVE = 5000

positive_pairs = []

valid_clusters = (
    deduplicated
    .groupby("cluster_id")
    .filter(lambda x: len(x) >= 2)
    .groupby("cluster_id")
)

for cluster_id, group in tqdm(valid_clusters, desc="Positive pairs"):
    idxs = group.index.to_list()

    if len(idxs) > 50:
        idxs = rng.choice(idxs, size=50, replace=False).tolist()

    pairs = list(combinations(idxs, 2))
    positive_pairs.extend(pairs)

positive_pairs = pd.DataFrame(
    positive_pairs,
    columns=["idx1", "idx2"]
)

positive_pairs["label"] = 1

positive_pairs = positive_pairs.sample(
    min(N_POSITIVE, len(positive_pairs)),
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("Positive pairs:", len(positive_pairs))
positive_pairs.head()

Positive pairs:   0%|          | 0/645892 [00:00<?, ?it/s]

Positive pairs: 5000


,idx1,idx2,label
0,2391291,3532769,1
1,136616,111900,1
2,170232,301174,1
3,163123,362802,1
4,2862349,2862351,1


In [4]:
# CELL 4 — random negative pairs

N_RANDOM_NEGATIVE = 5000

records = deduplicated.reset_index().rename(columns={"index": "orig_idx"})

negative_pairs = []

while len(negative_pairs) < N_RANDOM_NEGATIVE:
    sample = records.sample(
        2,
        random_state=int(rng.integers(0, 1_000_000))
    )

    r1, r2 = sample.iloc[0], sample.iloc[1]

    if r1["cluster_id"] != r2["cluster_id"]:
        negative_pairs.append((r1["orig_idx"], r2["orig_idx"]))

negative_pairs = pd.DataFrame(
    negative_pairs,
    columns=["idx1", "idx2"]
)

negative_pairs["label"] = 0

print("Random negative pairs:", len(negative_pairs))
negative_pairs.head()

Random negative pairs: 5000


,idx1,idx2,label
0,3326998,1479698,0
1,3324552,1292844,0
2,3504607,1067711,0
3,2045608,2876972,0
4,3313500,1523078,0


In [5]:
# CELL 5 — hard negative pairs by same country + similar prefix

N_HARD_NEGATIVE = 5000

tmp = deduplicated.copy()
tmp["prefix4"] = tmp["name_latin"].fillna("").str[:4]

hard_negatives = []

groups = tmp.groupby(["country", "prefix4"])

for _, group in tqdm(groups, desc="Hard negatives"):
    if len(hard_negatives) >= N_HARD_NEGATIVE:
        break

    if len(group) < 2:
        continue

    sampled = group.sample(
        min(len(group), 30),
        random_state=int(rng.integers(0, 1_000_000))
    )

    rows = sampled.reset_index()

    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            if rows.loc[i, "cluster_id"] != rows.loc[j, "cluster_id"]:
                hard_negatives.append(
                    (rows.loc[i, "index"], rows.loc[j, "index"])
                )

            if len(hard_negatives) >= N_HARD_NEGATIVE:
                break

        if len(hard_negatives) >= N_HARD_NEGATIVE:
            break

hard_negative_pairs = pd.DataFrame(
    hard_negatives,
    columns=["idx1", "idx2"]
)

hard_negative_pairs["label"] = 0

print("Hard negative pairs:", len(hard_negative_pairs))
hard_negative_pairs.head()

Hard negatives:   0%|          | 0/56445 [00:00<?, ?it/s]

Hard negative pairs: 5000


,idx1,idx2,label
0,372005,130158,0
1,372005,445926,0
2,372005,440912,0
3,372005,271125,0
4,372005,288085,0


In [6]:
# CELL 6 — final validation pairs

validation_pairs = pd.concat(
    [
        positive_pairs,
        negative_pairs,
        hard_negative_pairs,
    ],
    ignore_index=True
)

validation_pairs = validation_pairs.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

validation_pairs.to_parquet(
    DATA_DIR / "validation_pairs.parquet",
    index=False
)

print(validation_pairs["label"].value_counts())
validation_pairs.head()

label
0    10000
1     5000
Name: count, dtype: int64


,idx1,idx2,label
0,447734,73098,0
1,427003,397136,0
2,5649,440809,0
3,2272628,2309453,1
4,2733784,904015,0


In [7]:
# CELL 7 — attach record data for manual inspection

left = deduplicated.add_prefix("left_")
right = deduplicated.add_prefix("right_")

validation_view = (
    validation_pairs
    .merge(
        left,
        left_on="idx1",
        right_index=True,
        how="left"
    )
    .merge(
        right,
        left_on="idx2",
        right_index=True,
        how="left"
    )
)

cols = [
    "idx1", "idx2", "label",
    "left_country", "right_country",
    "left_party_name", "right_party_name",
    "left_name_latin", "right_name_latin",
    "left_cluster_id", "right_cluster_id",
]

validation_view[cols].to_csv(
    DATA_DIR / "validation_pairs_for_manual_review.csv",
    index=False
)

display(validation_view[cols].head(20))

,idx1,idx2,label,left_country,right_country,left_party_name,right_party_name,left_name_latin,right_name_latin,left_cluster_id,right_cluster_id
0,447734,73098,0,arm,arm,Աղասի Մքոյան,Աղասի Գրիգորյան,aghasi mqoyan,aghasi grigoryan,5506,18796
1,427003,397136,0,arm,arm,Դավիթ Ջալալյան,«ՔԱՖԵՎԻ»,davit jalalyan,qafevi,16981,17397
2,5649,440809,0,arm,arm,Ալեքսանդր Նաումով,Ալեքսանդր Տրունով,aleqsandr naoumov,aleqsandr trounov,3291,173861
3,2272628,2309453,1,kgz,kgz,Бабаев Расулбек Махмудович,Бабаев Расулбек Махмудович,babaev rasulbek mahmudovich,babaev rasulbek mahmudovich,1172920,1172920
4,2733784,904015,0,mng,kaz,дагий цогтбаяр,АНТРОПОВ АНДРЕЙ НИКОЛАЕВИЧ,dagii tsogtbayar,antropov andrei nikolaevich,1459645,436273
5,880357,973532,0,kaz,kaz,ДУСКАЛИЕВА АЙГУЛЬ РАМАЗАНОВНА,АМАНТАЕВА АНЕЛЯ АЙБЕКҚЫЗЫ,duskalieva ai gul ramazanovna,amantaeva anelya ai bekkyzy,474438,376912
6,2402720,2882864,1,mng,mng,Нацагдорж Соёлмаа,нацагдорж соёлмаа,natsagdorzh soe lmaa,natsagdorzh soe lmaa,1301758,1301758
7,1974774,1627800,1,kgz,kgz,Ли Юань,Ли Юн ---,li yuan,li yun,1000659,1000659
8,246875,296854,1,arm,arm,Նոննա Քոչարյան,Նոննա Քոչարյան,nonna qocharyan,nonna qocharyan,7933,7933
9,1753237,2107912,1,kgz,kgz,Дуйшоев Орозали Эркинович,Дуйшоев Орозали Эркинович,dui shoev orozali erkinovich,dui shoev orozali erkinovich,1127742,1127742


In [8]:
# CELL 8 — cluster distribution report

cluster_report = {
    "records_total": len(deduplicated),
    "entities_total": deduplicated["cluster_id"].nunique(),
    "duplicates_removed": len(deduplicated) - deduplicated["cluster_id"].nunique(),
    "max_cluster_size": int(cluster_stats["cluster_size"].max()),
    "mean_cluster_size": float(cluster_stats["cluster_size"].mean()),
    "median_cluster_size": float(cluster_stats["cluster_size"].median()),
    "suspicious_clusters": int(cluster_stats["is_suspicious"].sum()),
}

cluster_report

{'records_total': 3653581,
 'entities_total': 1709377,
 'duplicates_removed': 1944204,
 'max_cluster_size': 493,
 'mean_cluster_size': 2.137375780766911,
 'median_cluster_size': 1.0,
 'suspicious_clusters': 8138}

In [9]:
# CELL 9 — save cluster stats

cluster_stats.to_parquet(
    DATA_DIR / "cluster_stats.parquet",
    index=False
)

cluster_stats.to_csv(
    DATA_DIR / "cluster_stats.csv",
    index=False
)

print("Saved cluster_stats")

Saved cluster_stats


In [10]:
# CELL 10 — inspect suspicious clusters

top_suspicious = (
    cluster_stats
    .sort_values(
        ["is_suspicious", "cluster_size"],
        ascending=[False, False]
    )
    .head(10)["cluster_id"]
    .tolist()
)

for cid in top_suspicious[:5]:
    print("=" * 100)
    print("cluster_id:", cid)

    display(
        deduplicated[deduplicated["cluster_id"] == cid][
            ["country", "party_name", "name_latin", "cluster_id"]
        ]
        .drop_duplicates()
        .head(30)
    )

cluster_id: 376991


,country,party_name,name_latin,cluster_id
777198,kaz,ЕПАНЕШНИКОВА ИРИНА АЛЕКСАНДРОВНА,epaneshnikova irina aleksandrovna,376991


cluster_id: 3110


,country,party_name,name_latin,cluster_id
5360,arm,Վաչե Մանուկյան,vache manoukyan,3110


cluster_id: 165


,country,party_name,name_latin,cluster_id
171,arm,Սարգիս Կարապետյան,sargis karapetyan,165
36259,arm,ՍԱՐԳԻՍ ԿԱՐԱՊԵՏՅԱՆ,sargis karapetyan,165


cluster_id: 305


,country,party_name,name_latin,cluster_id
319,arm,ՎԵՐՈՆԻԿԱ ԶՈՆԱԲԵՆԴ,veronika zonabend,305
15338,arm,Վերոնիկա Զոնաբենդ,veronika zonabend,305


cluster_id: 174


,country,party_name,name_latin,cluster_id
180,arm,Վահե Պետրոսյան,vahe petrosyan,174


In [11]:
manual_review = validation_view[
    [
        "label",
        "left_party_name",
        "right_party_name",
        "left_name_latin",
        "right_name_latin",
        "left_country",
        "right_country",
    ]
].sample(
    300,
    random_state=42,
)

manual_review.to_csv(
    "../data/full_dedup/manual_review.csv",
    index=False,
)

manual_review.head(20)

,label,left_party_name,right_party_name,left_name_latin,right_name_latin,left_country,right_country
11499,0,ДАЦКО ИЛЬЯ СЕРГЕЕВИЧ,Անհայտ կազմակերպություն,datsko ilya sergeevich,anhayt kazmakerpoutyoun,kaz,arm
6475,1,Бекмамбетова Кымбат Шааматовна,Бекмамбетова Кымбат Шааматовна,bekmambetova kymbat shaamatovna,bekmambetova kymbat shaamatovna,kgz,kgz
13167,0,Jeon Ho Cheol,KƏRİMOVA NAİBƏ FAZİL QIZI,jeon ho cheol,keri mova nai be fazi l qizi,mng,azb
862,1,Լյուդվիկ Տիգրանյան,Լյուդվիկ Տիգրանյան,lyoudvik tigranyan,lyoudvik tigranyan,arm,arm
5970,1,сэрод цэцэгмаа,цэцэгмаа сэрод,serod tsetsegmaa,tsetsegmaa serod,mng,mng
6706,1,АРТЫКБАЕВА ДАМИРА БАРАТОВНА,АРТЫКБАЕВА ДАМИРА БАРАТОВНА,artykbaeva damira baratovna,artykbaeva damira baratovna,kaz,kaz
3017,0,адъяа цолмон,Կարեն Հովհաննիսյան,adyaa tsolmon,karen hovhannisyan,mng,arm
3781,0,Ադել Մուհամմադ Ալի Ջասիմ Ալ Մարզուqի,Ադել Մուհամեդ Ալի Ջասիմ Ալ Մարզուգի,adel mouhammad ali jasim al marzouqi,adel mouhamed ali jasim al marzougi,arm,arm
3898,0,VƏLİYEV ƏDALƏT MUXTAR OĞLU,Оюунбат Балтбаатар,veli yev edalet muxtar og lu,oyuunbat baltbaatar,azb,mng
2250,1,ƏHMƏDOV AZƏR İBRAHİM OĞLU,ƏHMƏDOV AZƏR İBRAHİM OĞLU,ehmedov azer i brahi m og lu,ehmedov azer i brahi m og lu,azb,azb
